# Case Study 5: Multi-class Text Classification — AG News

## RNN vs LSTM vs GRU Comprehensive Comparison

---

### Objective
Classify news articles into **4 categories** (World, Sports, Business, Sci/Tech) using three recurrent architectures:
- **Vanilla RNN** — simple recurrence, prone to vanishing gradients on long text
- **LSTM** — gated memory cells (forget, input, output gates) for long-range dependencies
- **GRU** — simplified gating (reset, update gates), fewer parameters than LSTM

### What You Will Learn
1. How to frame multi-class text classification as a sequence problem
2. Building a vocabulary from scratch with frequency-based filtering
3. Embedding + RNN/LSTM/GRU architectures for NLP
4. Proper evaluation with macro/weighted F1, confusion matrices, per-class analysis
5. Hyperparameter tuning with random search on a subsampled dataset
6. Comparing against a TF-IDF + Naive Bayes baseline

### Dataset
- **AG News**: 120,000 training + 7,600 test news articles across 4 balanced classes
- Classes: 1=World, 2=Sports, 3=Business, 4=Sci/Tech
- Each sample has a title and description
- A step up from binary classification — requires the model to distinguish between 4 topics

### Why Multi-class Matters
Binary classification (positive/negative) is a simpler problem where a single threshold suffices. Multi-class classification introduces:
- **Softmax** output layer instead of sigmoid
- **CrossEntropyLoss** instead of BCELoss
- Per-class evaluation metrics (precision, recall, F1 for each class)
- Confusion matrix analysis to understand inter-class misclassifications
- Macro vs weighted averaging for imbalanced evaluation

---
## 1. Environment Setup

In [ ]:
import random
import time
import string
import warnings
warnings.filterwarnings('ignore')
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Plot style ────────────────────────────────────────────────────
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
COLORS = {'RNN': '#e74c3c', 'LSTM': '#2ecc71', 'GRU': '#3498db'}

# ── Class mapping ────────────────────────────────────────────────
CLASS_NAMES = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
CLASS_LIST = ['World', 'Sports', 'Business', 'Sci/Tech']
NUM_CLASSES = 4
print(f'\nClass mapping: {CLASS_NAMES}')

---
## 2. Exploratory Data Analysis

In [ ]:
# Load datasets — CSV with NO header, 3 columns
train_df = pd.read_csv('Text_and_NLP/ag_news/ag_news_train.csv',
                       header=None, names=['label', 'title', 'description'])
test_df = pd.read_csv('Text_and_NLP/ag_news/ag_news_test.csv',
                      header=None, names=['label', 'title', 'description'])

print(f'Training set:  {train_df.shape[0]:,} samples')
print(f'Test set:      {test_df.shape[0]:,} samples')
print(f'Total:         {len(train_df) + len(test_df):,} samples')
print(f'\nLabel values: {sorted(train_df["label"].unique())}')
print(f'Missing values (train):\n{train_df.isnull().sum()}')
print(f'\nMissing values (test):\n{test_df.isnull().sum()}')
train_df.head(10)

In [ ]:
# Fill any NaN text fields with empty string
for col in ['title', 'description']:
    train_df[col] = train_df[col].fillna('')
    test_df[col] = test_df[col].fillna('')

# Add class name column for easier analysis
train_df['class_name'] = train_df['label'].map(CLASS_NAMES)
test_df['class_name'] = test_df['label'].map(CLASS_NAMES)

# Text length features
train_df['title_len'] = train_df['title'].str.len()
train_df['desc_len'] = train_df['description'].str.len()
train_df['title_words'] = train_df['title'].str.split().str.len()
train_df['desc_words'] = train_df['description'].str.split().str.len()
train_df['total_text'] = train_df['title'] + ' ' + train_df['description']
train_df['total_len'] = train_df['total_text'].str.len()
train_df['total_words'] = train_df['total_text'].str.split().str.len()

print('Text length statistics (characters):')
train_df[['title_len', 'desc_len', 'total_len']].describe().round(1)

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
class_counts_train = train_df['class_name'].value_counts().reindex(CLASS_LIST)
bars = axes[0].bar(CLASS_LIST, class_counts_train.values,
                   color=['#e74c3c', '#2ecc71', '#3498db', '#9b59b6'], edgecolor='white')
for bar, val in zip(bars, class_counts_train.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('Training Set Class Distribution', fontsize=13)
axes[0].set_ylabel('Number of Samples')
axes[0].set_ylim(0, class_counts_train.max() * 1.12)

# Test set
class_counts_test = test_df['class_name'].value_counts().reindex(CLASS_LIST)
bars = axes[1].bar(CLASS_LIST, class_counts_test.values,
                   color=['#e74c3c', '#2ecc71', '#3498db', '#9b59b6'], edgecolor='white')
for bar, val in zip(bars, class_counts_test.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('Test Set Class Distribution', fontsize=13)
axes[1].set_ylabel('Number of Samples')
axes[1].set_ylim(0, class_counts_test.max() * 1.12)

plt.suptitle('AG News — Perfectly Balanced Classes', fontsize=14)
plt.tight_layout()
plt.show()

print('Train class counts:')
for cls, cnt in class_counts_train.items():
    print(f'  {cls}: {cnt:,} ({cnt/len(train_df)*100:.1f}%)')

In [ ]:
# Text length distributions per class
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
class_colors = {'World': '#e74c3c', 'Sports': '#2ecc71', 'Business': '#3498db', 'Sci/Tech': '#9b59b6'}

for ax, cls_name in zip(axes.flat, CLASS_LIST):
    subset = train_df[train_df['class_name'] == cls_name]
    ax.hist(subset['total_words'], bins=50, color=class_colors[cls_name],
            alpha=0.75, edgecolor='white', density=True)
    mean_len = subset['total_words'].mean()
    median_len = subset['total_words'].median()
    ax.axvline(mean_len, color='black', linestyle='--', linewidth=1.5, label=f'Mean: {mean_len:.0f}')
    ax.axvline(median_len, color='gray', linestyle=':', linewidth=1.5, label=f'Median: {median_len:.0f}')
    ax.set_title(f'{cls_name} — Word Count Distribution', fontsize=12)
    ax.set_xlabel('Number of Words')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle('Text Length Distribution by Class (Title + Description)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Sample articles from each class
print('=' * 90)
print('SAMPLE ARTICLES FROM EACH CLASS')
print('=' * 90)

for cls_label, cls_name in CLASS_NAMES.items():
    print(f'\n{"─" * 90}')
    print(f'CLASS {cls_label}: {cls_name.upper()}')
    print(f'{"─" * 90}')
    samples = train_df[train_df['label'] == cls_label].sample(3, random_state=SEED)
    for _, row in samples.iterrows():
        print(f'  Title: {row["title"][:100]}')
        print(f'  Desc:  {row["description"][:150]}...')
        print()

In [ ]:
# Word frequency analysis per class
from collections import Counter

def get_top_words(texts, n=20):
    """Get top-n most common words from a series of texts."""
    all_words = []
    for text in texts:
        words = str(text).lower().translate(str.maketrans('', '', string.punctuation)).split()
        all_words.extend(words)
    return Counter(all_words).most_common(n)

# Stopwords to exclude (simple list)
STOPWORDS = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'in', 'on', 'at', 'to', 'for',
             'of', 'and', 'or', 'but', 'not', 'with', 'by', 'from', 'as', 'it', 'its',
             'that', 'this', 'has', 'have', 'had', 'be', 'been', 'will', 'would', 'could',
             'should', 'may', 'can', 'do', 'did', 'does', 'he', 'she', 'they', 'we', 'i',
             'you', 'my', 'his', 'her', 'their', 'our', 'your', 'said', 'also', 'about',
             'more', 'than', 'who', 'which', 'when', 'what', 'where', 'how', 'all', 'each',
             'no', 'so', 'up', 'out', 'if', 'into', 'over', 'after', 'just', 'new', 'one',
             'two', 'been', 'first', 'last', 'own', 'other', 'some', 'there', 'most', 'only',
             'very', 'even', 'back', 'get', 'make', 'like', 'well', 'much', 'then', 'any',
             'between', 'before', 'because', 'while', 'during', 'through', 'under', 'being',
             'those', 'both', 'these', 'way', 'them', 'him', 'down', 'now', 'here', 'still',
             'going', 'many', 'off', 'set', 'came', 'take', 'took', 'come', 'went', 'made',
             'such', 'year', 'years', 'quot', 'since', 'us', 'lt', 'gt', 'amp', 'ap',
             'reuters', 'afp'}

def get_top_words_filtered(texts, n=15):
    """Get top words excluding stopwords."""
    all_words = []
    for text in texts:
        words = str(text).lower().translate(str.maketrans('', '', string.punctuation)).split()
        words = [w for w in words if w not in STOPWORDS and len(w) > 2]
        all_words.extend(words)
    return Counter(all_words).most_common(n)


fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, cls_name in zip(axes.flat, CLASS_LIST):
    subset = train_df[train_df['class_name'] == cls_name]
    top_words = get_top_words_filtered(subset['total_text'], n=15)
    words, counts = zip(*top_words)
    ax.barh(range(len(words)), counts, color=class_colors[cls_name], edgecolor='white')
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=10)
    ax.invert_yaxis()
    ax.set_title(f'{cls_name} — Top 15 Words (excl. stopwords)', fontsize=12)
    ax.set_xlabel('Frequency')

plt.suptitle('Most Frequent Words per Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Title length vs Description length
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Boxplot of title word counts by class
data_title = [train_df[train_df['class_name'] == c]['title_words'].values for c in CLASS_LIST]
bp1 = axes[0].boxplot(data_title, labels=CLASS_LIST, patch_artist=True, showfliers=False)
for patch, color in zip(bp1['boxes'], [class_colors[c] for c in CLASS_LIST]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('Title Word Count by Class', fontsize=13)
axes[0].set_ylabel('Word Count')

# Boxplot of description word counts by class
data_desc = [train_df[train_df['class_name'] == c]['desc_words'].values for c in CLASS_LIST]
bp2 = axes[1].boxplot(data_desc, labels=CLASS_LIST, patch_artist=True, showfliers=False)
for patch, color in zip(bp2['boxes'], [class_colors[c] for c in CLASS_LIST]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Description Word Count by Class', fontsize=13)
axes[1].set_ylabel('Word Count')

plt.suptitle('Title vs Description Length Comparison', fontsize=14)
plt.tight_layout()
plt.show()

# Stats table
print('Average word counts by class:')
for cls_name in CLASS_LIST:
    subset = train_df[train_df['class_name'] == cls_name]
    print(f'  {cls_name:10s} | Title: {subset["title_words"].mean():.1f} words | '
          f'Description: {subset["desc_words"].mean():.1f} words | '
          f'Total: {subset["total_words"].mean():.1f} words')

In [ ]:
# Title vs Description length scatter (sampled for speed)
fig, ax = plt.subplots(figsize=(10, 7))

for cls_name in CLASS_LIST:
    subset = train_df[train_df['class_name'] == cls_name].sample(2000, random_state=SEED)
    ax.scatter(subset['title_words'], subset['desc_words'],
               alpha=0.15, s=10, color=class_colors[cls_name], label=cls_name)

ax.set_xlabel('Title Word Count', fontsize=12)
ax.set_ylabel('Description Word Count', fontsize=12)
ax.set_title('Title vs Description Length by Class', fontsize=14)
ax.legend(fontsize=11, markerscale=3)
plt.tight_layout()
plt.show()

print('Observations:')
print('- All 4 classes are perfectly balanced (30K samples each in train)')
print('- Text lengths are fairly similar across classes')
print('- Each class has distinct vocabulary: sports terms, financial terms, tech terms, etc.')
print('- Descriptions are significantly longer than titles (avg ~30 vs ~8 words)')

---
## 3. Text Preprocessing

**Pipeline:**
1. Combine title + description into a single text field
2. Lowercase, remove punctuation, split tokenization
3. Build vocabulary with PAD=0, UNK=1, min_freq=2
4. Convert tokens to integer indices
5. Pad/truncate sequences to MAX_SEQ_LEN=200
6. Split training data into 90% train / 10% validation
7. Create PyTorch Datasets and DataLoaders

In [ ]:
MAX_SEQ_LEN = 200
MIN_FREQ = 2
BATCH_SIZE = 64

def preprocess_text(text):
    """Lowercase, remove punctuation, tokenize by whitespace."""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    return tokens

# Combine title + description
train_df['text'] = train_df['title'] + ' ' + train_df['description']
test_df['text'] = test_df['title'] + ' ' + test_df['description']

# Tokenize
train_df['tokens'] = train_df['text'].apply(preprocess_text)
test_df['tokens'] = test_df['text'].apply(preprocess_text)

print(f'Sample tokenization:')
print(f'  Original: {train_df["text"].iloc[0][:100]}...')
print(f'  Tokens:   {train_df["tokens"].iloc[0][:15]}...')
print(f'\nToken count stats:')
print(f'  Mean:   {train_df["tokens"].str.len().mean():.1f}')
print(f'  Median: {train_df["tokens"].str.len().median():.0f}')
print(f'  Max:    {train_df["tokens"].str.len().max()}')
print(f'  95th percentile: {train_df["tokens"].str.len().quantile(0.95):.0f}')

In [ ]:
# Build vocabulary from training data only
PAD_IDX = 0
UNK_IDX = 1

def build_vocab(token_lists, min_freq=2):
    """Build word-to-index vocabulary with PAD=0, UNK=1.
    
    Args:
        token_lists: iterable of token lists
        min_freq: minimum frequency to include a word
    
    Returns:
        word2idx: dict mapping word -> index
        idx2word: dict mapping index -> word
    """
    word_counts = Counter()
    for tokens in token_lists:
        word_counts.update(tokens)
    
    word2idx = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
    idx = 2
    for word, count in word_counts.most_common():
        if count >= min_freq:
            word2idx[word] = idx
            idx += 1
    
    idx2word = {v: k for k, v in word2idx.items()}
    return word2idx, idx2word


word2idx, idx2word = build_vocab(train_df['tokens'], min_freq=MIN_FREQ)
VOCAB_SIZE = len(word2idx)

# Count how many words were filtered out
all_train_words = Counter()
for tokens in train_df['tokens']:
    all_train_words.update(tokens)
total_unique = len(all_train_words)
filtered_out = total_unique - (VOCAB_SIZE - 2)  # subtract PAD and UNK

print(f'Vocabulary Statistics:')
print(f'  Total unique words in training data: {total_unique:,}')
print(f'  Words with freq >= {MIN_FREQ}: {VOCAB_SIZE - 2:,}')
print(f'  Filtered out (freq < {MIN_FREQ}): {filtered_out:,}')
print(f'  Final vocab size (incl PAD, UNK): {VOCAB_SIZE:,}')
print(f'\n  PAD index: {PAD_IDX}')
print(f'  UNK index: {UNK_IDX}')
print(f'  Sample mappings: "the" -> {word2idx.get("the", "N/A")}, '
      f'"sports" -> {word2idx.get("sports", "N/A")}, '
      f'"technology" -> {word2idx.get("technology", "N/A")}')

In [ ]:
def encode_tokens(token_list, word2idx, max_len):
    """Convert tokens to integer indices with padding/truncation."""
    indices = [word2idx.get(token, UNK_IDX) for token in token_list]
    # Truncate
    if len(indices) > max_len:
        indices = indices[:max_len]
    # Pad
    else:
        indices = indices + [PAD_IDX] * (max_len - len(indices))
    return indices


# Encode all texts
train_encoded = [encode_tokens(tokens, word2idx, MAX_SEQ_LEN) for tokens in train_df['tokens']]
test_encoded = [encode_tokens(tokens, word2idx, MAX_SEQ_LEN) for tokens in test_df['tokens']]

# Convert labels: original labels are 1-4, convert to 0-3 for CrossEntropyLoss
train_labels = (train_df['label'].values - 1).astype(np.int64)
test_labels = (test_df['label'].values - 1).astype(np.int64)

# Convert to tensors
X_all_train = torch.tensor(train_encoded, dtype=torch.long)
y_all_train = torch.tensor(train_labels, dtype=torch.long)
X_test = torch.tensor(test_encoded, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.long)

print(f'Encoded shapes:')
print(f'  X_all_train: {X_all_train.shape}')
print(f'  y_all_train: {y_all_train.shape}  (labels: {sorted(y_all_train.unique().tolist())})')
print(f'  X_test:      {X_test.shape}')
print(f'  y_test:      {y_test.shape}')
print(f'\nSample encoded sequence (first 20 tokens): {X_all_train[0][:20].tolist()}')

In [ ]:
# Split train into 90% train / 10% validation (stratified)
from sklearn.model_selection import train_test_split

indices = np.arange(len(X_all_train))
train_idx, val_idx = train_test_split(
    indices, test_size=0.1, random_state=SEED, stratify=train_labels
)

X_train = X_all_train[train_idx]
y_train = y_all_train[train_idx]
X_val = X_all_train[val_idx]
y_val = y_all_train[val_idx]

print(f'Final splits:')
print(f'  Train:      {X_train.shape[0]:,} samples')
print(f'  Validation: {X_val.shape[0]:,} samples')
print(f'  Test:       {X_test.shape[0]:,} samples')

# Verify stratification
print(f'\nLabel distribution (train):     {np.bincount(y_train.numpy())}')
print(f'Label distribution (val):       {np.bincount(y_val.numpy())}')
print(f'Label distribution (test):      {np.bincount(y_test.numpy())}')

In [ ]:
# PyTorch Dataset and DataLoaders
class TextDataset(Dataset):
    """Simple text classification dataset."""
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = TextDataset(X_train, y_train)
val_dataset = TextDataset(X_val, y_val)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f'DataLoaders created (batch_size={BATCH_SIZE}):')
print(f'  Train: {len(train_loader)} batches')
print(f'  Val:   {len(val_loader)} batches')
print(f'  Test:  {len(test_loader)} batches')

# Verify a batch
sample_X, sample_y = next(iter(train_loader))
print(f'\nSample batch: X={sample_X.shape}, y={sample_y.shape}')
print(f'  X dtype: {sample_X.dtype}, y dtype: {sample_y.dtype}')

---
## 4. Model Architecture

### Architecture Diagram
```
Input: (batch, seq_len=200)     <- token indices
       |
  [Embedding Layer]              <- (vocab_size, embed_dim)
       |
  (batch, 200, embed_dim)       <- dense word vectors
       |
  +--------+   +--------+         +--------+
  | RNN /  |-->| RNN /  |--> ... ->| RNN /  |--> h_T
  | LSTM / |   | LSTM / |         | LSTM / |
  | GRU    |   | GRU    |         | GRU    |
  +--------+   +--------+         +--------+
    t=1          t=2                t=200
                                     |
                              [Dropout Layer]
                                     |
                              [Linear Layer]
                                     |
                              Output: (batch, 4)
                        (logits for 4 news classes)
```

### Key Differences from Binary Classification
- Output layer: `Linear(hidden, 4)` instead of `Linear(hidden, 1)`
- Loss: `CrossEntropyLoss` instead of `BCEWithLogitsLoss`
- Prediction: `argmax` over 4 logits instead of thresholding a single logit

In [ ]:
class TextClassifier(nn.Module):
    """Unified RNN/LSTM/GRU text classifier with Embedding layer.
    
    Architecture: Embedding -> RNN/LSTM/GRU -> Dropout -> Linear(hidden, num_classes)
    Uses the last hidden state for classification.
    Supports bidirectional mode (doubles the effective hidden size for the FC layer).
    """
    
    SUPPORTED_TYPES = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}
    
    def __init__(self, model_type, vocab_size, embed_dim, hidden_size, num_classes,
                 num_layers=1, dropout=0.0, bidirectional=False, pad_idx=0):
        super().__init__()
        assert model_type in self.SUPPORTED_TYPES, \
            f'model_type must be one of {list(self.SUPPORTED_TYPES.keys())}'
        
        self.model_type = model_type
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        # Embedding layer
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )
        
        # Recurrent layer
        rnn_cls = self.SUPPORTED_TYPES[model_type]
        self.rnn = rnn_cls(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        
        # Output layer
        self.dropout = nn.Dropout(dropout)
        fc_input_size = hidden_size * self.num_directions
        self.fc = nn.Linear(fc_input_size, num_classes)
    
    def forward(self, x):
        # x: (batch, seq_len) — token indices
        embedded = self.embedding(x)          # (batch, seq_len, embed_dim)
        rnn_out, _ = self.rnn(embedded)        # (batch, seq_len, hidden * num_directions)
        last_hidden = rnn_out[:, -1, :]        # (batch, hidden * num_directions)
        dropped = self.dropout(last_hidden)
        logits = self.fc(dropped)              # (batch, num_classes)
        return logits
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Compare parameter counts for default configuration
print('Parameter Count Comparison')
print(f'  vocab_size={VOCAB_SIZE:,}, embed_dim=128, hidden=128, layers=1, classes=4')
print('=' * 60)

param_comparison = []
for model_type in ['RNN', 'LSTM', 'GRU']:
    model = TextClassifier(
        model_type=model_type,
        vocab_size=VOCAB_SIZE,
        embed_dim=128,
        hidden_size=128,
        num_classes=NUM_CLASSES,
        num_layers=1
    )
    total_params = model.count_parameters()
    embed_params = sum(p.numel() for p in model.embedding.parameters())
    rnn_params = sum(p.numel() for p in model.rnn.parameters())
    fc_params = sum(p.numel() for p in model.fc.parameters())
    
    param_comparison.append({
        'Model': model_type,
        'Embedding': embed_params,
        'RNN/LSTM/GRU': rnn_params,
        'FC Layer': fc_params,
        'Total': total_params
    })
    print(f'  {model_type:5s}: Embed={embed_params:>10,} | RNN={rnn_params:>8,} | '
          f'FC={fc_params:>5,} | Total={total_params:>10,}')

param_df = pd.DataFrame(param_comparison)

print(f'\nNote: Embedding layer dominates parameter count ({embed_params:,} params).')
print(f'The RNN/LSTM/GRU layer parameter differences are relatively small.')

In [ ]:
# Visualize parameter breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total parameters
bars = axes[0].bar(['RNN', 'LSTM', 'GRU'],
                   [p['Total'] for p in param_comparison],
                   color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']], edgecolor='white')
for bar, val in zip(bars, [p['Total'] for p in param_comparison]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Total Parameters', fontsize=13)
axes[0].set_ylabel('Parameters')

# Recurrent layer only (excl. embedding)
rnn_only = [p['RNN/LSTM/GRU'] for p in param_comparison]
bars = axes[1].bar(['RNN', 'LSTM', 'GRU'], rnn_only,
                   color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']], edgecolor='white')
for bar, val in zip(bars, rnn_only):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=10)
axes[1].set_title('Recurrent Layer Parameters Only', fontsize=13)
axes[1].set_ylabel('Parameters')

plt.suptitle('Parameter Comparison: RNN vs LSTM vs GRU', fontsize=14)
plt.tight_layout()
plt.show()

print('LSTM has ~4x RNN recurrent params (3 extra gates).')
print('GRU has ~3x RNN recurrent params (2 gates).')
print('But embedding dominates, so total differences are small.')

In [ ]:
# Verify forward pass
for mt in ['RNN', 'LSTM', 'GRU']:
    model = TextClassifier(mt, VOCAB_SIZE, embed_dim=128, hidden_size=128,
                           num_classes=NUM_CLASSES, num_layers=1, bidirectional=True)
    model.eval()
    with torch.no_grad():
        out = model(sample_X)
    print(f'{mt} (bidirectional): input={sample_X.shape} -> output={out.shape}')

print(f'\nOutput shape is (batch={BATCH_SIZE}, num_classes={NUM_CLASSES}) as expected.')

---
## 5. Training Infrastructure

In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr,
                device=DEVICE, patience=5, clip_grad=1.0, verbose=True):
    """Train a multi-class text classifier with early stopping.
    
    Tracks: loss, accuracy, macro-F1 on both train and validation sets.
    Uses CrossEntropyLoss and Adam optimizer with gradient clipping.
    
    Returns:
        history: dict with train/val loss, accuracy, macro_f1
        training_time: total wall-clock time in seconds
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=3, factor=0.5, verbose=False
    )
    
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'train_f1': [], 'val_f1': []
    }
    
    best_val_f1 = 0.0
    best_state = None
    patience_counter = 0
    
    start_time = time.time()
    
    for epoch in range(epochs):
        # ── Training ──
        model.train()
        train_losses = []
        all_train_preds = []
        all_train_labels = []
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            
            train_losses.append(loss.item())
            preds = logits.argmax(dim=1).cpu().numpy()
            all_train_preds.extend(preds)
            all_train_labels.extend(y_batch.cpu().numpy())
        
        train_loss = np.mean(train_losses)
        train_acc = accuracy_score(all_train_labels, all_train_preds)
        train_f1 = f1_score(all_train_labels, all_train_preds, average='macro')
        
        # ── Validation ──
        model.eval()
        val_losses = []
        all_val_preds = []
        all_val_labels = []
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_losses.append(loss.item())
                preds = logits.argmax(dim=1).cpu().numpy()
                all_val_preds.extend(preds)
                all_val_labels.extend(y_batch.cpu().numpy())
        
        val_loss = np.mean(val_losses)
        val_acc = accuracy_score(all_val_labels, all_val_preds)
        val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')
        
        # Record
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)
        
        scheduler.step(val_f1)
        
        # Early stopping on val F1
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch+1} (best val F1: {best_val_f1:.4f})')
                break
        
        if verbose and (epoch + 1) % 2 == 0:
            print(f'  Epoch {epoch+1:3d}/{epochs} | '
                  f'Loss: {train_loss:.4f}/{val_loss:.4f} | '
                  f'Acc: {train_acc:.4f}/{val_acc:.4f} | '
                  f'F1: {train_f1:.4f}/{val_f1:.4f}')
    
    training_time = time.time() - start_time
    
    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)
        model = model.to(device)
    
    return history, training_time

In [ ]:
def evaluate_model(model, data_loader, device=DEVICE):
    """Evaluate model on a dataset. Returns predictions, labels, and metrics."""
    model = model.to(device)
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())
            all_probs.extend(probs)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    metrics = {
        'Accuracy': accuracy_score(all_labels, all_preds),
        'Macro-F1': f1_score(all_labels, all_preds, average='macro'),
        'Weighted-F1': f1_score(all_labels, all_preds, average='weighted'),
        'Macro-Precision': precision_score(all_labels, all_preds, average='macro'),
        'Macro-Recall': recall_score(all_labels, all_preds, average='macro'),
    }
    
    return all_preds, all_labels, all_probs, metrics

In [ ]:
# Quick sanity check: train 1 epoch with a small model
print('Sanity check: 1 epoch with small LSTM...')
sanity_model = TextClassifier('LSTM', VOCAB_SIZE, embed_dim=64, hidden_size=64,
                               num_classes=NUM_CLASSES, num_layers=1, dropout=0.0)
sanity_history, sanity_time = train_model(
    sanity_model, train_loader, val_loader, epochs=1, lr=1e-3, verbose=False
)
print(f'  Train loss: {sanity_history["train_loss"][0]:.4f}')
print(f'  Val acc:    {sanity_history["val_acc"][0]:.4f}')
print(f'  Val F1:     {sanity_history["val_f1"][0]:.4f}')
print(f'  Time:       {sanity_time:.1f}s')
print(f'  Random baseline accuracy would be ~0.25 (4 classes).')
print(f'  Model is learning!' if sanity_history['val_acc'][0] > 0.5 else '  Check model setup.')
del sanity_model

---
## 6. Hyperparameter Tuning

We subsample to 30K training examples and perform 10 random trials per model type to find the best configurations.

**Search space:**
- `hidden_size`: [64, 128, 256]
- `num_layers`: [1, 2]
- `lr`: [5e-4, 1e-3, 5e-3]
- `embed_dim`: [64, 128]
- `dropout`: [0.0, 0.2, 0.3]
- `bidirectional`: [True, False]

In [ ]:
# Subsample training data for faster hyperparameter search
SUBSAMPLE_SIZE = 30000
TUNING_EPOCHS = 5
N_TRIALS = 10

sub_indices = np.random.choice(len(X_train), size=SUBSAMPLE_SIZE, replace=False)
X_train_sub = X_train[sub_indices]
y_train_sub = y_train[sub_indices]

sub_dataset = TextDataset(X_train_sub, y_train_sub)
sub_loader = DataLoader(sub_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f'Subsampled training set: {len(sub_dataset):,} samples')
print(f'Tuning epochs: {TUNING_EPOCHS}')
print(f'Trials per model: {N_TRIALS}')
print(f'Total trials: {N_TRIALS * 3}')

SEARCH_SPACE = {
    'hidden_size': [64, 128, 256],
    'num_layers': [1, 2],
    'lr': [5e-4, 1e-3, 5e-3],
    'embed_dim': [64, 128],
    'dropout': [0.0, 0.2, 0.3],
    'bidirectional': [True, False],
}

print(f'\nSearch space:')
for k, v in SEARCH_SPACE.items():
    print(f'  {k}: {v}')

In [ ]:
tuning_results = []

for model_type in ['RNN', 'LSTM', 'GRU']:
    print(f'\n{"="*70}')
    print(f'Tuning {model_type} ({N_TRIALS} random trials)')
    print(f'{"="*70}')
    
    for trial in range(N_TRIALS):
        config = {k: random.choice(v) for k, v in SEARCH_SPACE.items()}
        
        model = TextClassifier(
            model_type=model_type,
            vocab_size=VOCAB_SIZE,
            embed_dim=config['embed_dim'],
            hidden_size=config['hidden_size'],
            num_classes=NUM_CLASSES,
            num_layers=config['num_layers'],
            dropout=config['dropout'],
            bidirectional=config['bidirectional']
        )
        
        history, t = train_model(
            model, sub_loader, val_loader,
            epochs=TUNING_EPOCHS, lr=config['lr'], verbose=False, patience=3
        )
        
        best_val_f1 = max(history['val_f1'])
        best_val_acc = max(history['val_acc'])
        
        tuning_results.append({
            'model_type': model_type,
            'trial': trial,
            **config,
            'best_val_f1': best_val_f1,
            'best_val_acc': best_val_acc,
            'train_time': t,
            'epochs_run': len(history['val_f1']),
            'params': model.count_parameters()
        })
        
        bidir_str = 'Bi' if config['bidirectional'] else 'Uni'
        print(f'  Trial {trial+1:2d}/{N_TRIALS}: h={config["hidden_size"]:3d}, '
              f'L={config["num_layers"]}, e={config["embed_dim"]:3d}, '
              f'd={config["dropout"]:.1f}, {bidir_str}, '
              f'lr={config["lr"]:.4f} -> F1={best_val_f1:.4f} ({t:.1f}s)')
        
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

tuning_df = pd.DataFrame(tuning_results)
print(f'\nTotal trials completed: {len(tuning_df)}')

In [ ]:
# Find best configuration for each model type
best_configs = {}
print('Best Configurations per Model Type')
print('=' * 80)

for model_type in ['RNN', 'LSTM', 'GRU']:
    subset = tuning_df[tuning_df['model_type'] == model_type]
    best_row = subset.loc[subset['best_val_f1'].idxmax()]
    best_configs[model_type] = best_row.to_dict()
    
    bidir_str = 'Bidirectional' if best_row['bidirectional'] else 'Unidirectional'
    print(f'\n{model_type}:')
    print(f'  hidden_size={int(best_row["hidden_size"])}, '
          f'num_layers={int(best_row["num_layers"])}, '
          f'embed_dim={int(best_row["embed_dim"])}, '
          f'dropout={best_row["dropout"]:.1f}, '
          f'{bidir_str}')
    print(f'  lr={best_row["lr"]}, best_val_f1={best_row["best_val_f1"]:.4f}, '
          f'params={int(best_row["params"]):,}')

In [ ]:
# Visualize tuning results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model_type in zip(axes, ['RNN', 'LSTM', 'GRU']):
    subset = tuning_df[tuning_df['model_type'] == model_type].sort_values('best_val_f1', ascending=False)
    trials = range(1, len(subset) + 1)
    bars = ax.bar(trials, subset['best_val_f1'].values,
                  color=COLORS[model_type], edgecolor='white', alpha=0.8)
    # Highlight best
    bars[0].set_edgecolor('black')
    bars[0].set_linewidth(2)
    ax.set_title(f'{model_type} — Trials Ranked by Val F1', fontsize=12)
    ax.set_xlabel('Trial Rank')
    ax.set_ylabel('Best Val Macro-F1')
    ax.set_ylim(subset['best_val_f1'].min() - 0.02, subset['best_val_f1'].max() + 0.02)

plt.suptitle('Hyperparameter Tuning Results (10 Trials per Model)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Impact of bidirectional vs unidirectional
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model_type in zip(axes, ['RNN', 'LSTM', 'GRU']):
    subset = tuning_df[tuning_df['model_type'] == model_type]
    uni = subset[subset['bidirectional'] == False]['best_val_f1']
    bi = subset[subset['bidirectional'] == True]['best_val_f1']
    
    data_to_plot = []
    labels_to_plot = []
    if len(uni) > 0:
        data_to_plot.append(uni.values)
        labels_to_plot.append('Unidirectional')
    if len(bi) > 0:
        data_to_plot.append(bi.values)
        labels_to_plot.append('Bidirectional')
    
    bp = ax.boxplot(data_to_plot, labels=labels_to_plot, patch_artist=True, showmeans=True)
    for patch in bp['boxes']:
        patch.set_facecolor(COLORS[model_type])
        patch.set_alpha(0.6)
    ax.set_title(f'{model_type}', fontsize=13)
    ax.set_ylabel('Val Macro-F1')

plt.suptitle('Bidirectional vs Unidirectional Performance', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Impact of hidden size on performance
fig, ax = plt.subplots(figsize=(10, 5))

for model_type in ['RNN', 'LSTM', 'GRU']:
    subset = tuning_df[tuning_df['model_type'] == model_type]
    grouped = subset.groupby('hidden_size')['best_val_f1'].mean()
    ax.plot(grouped.index, grouped.values, 'o-', color=COLORS[model_type],
            label=model_type, linewidth=2, markersize=8)

ax.set_xlabel('Hidden Size', fontsize=12)
ax.set_ylabel('Mean Val Macro-F1', fontsize=12)
ax.set_title('Effect of Hidden Size on Performance', fontsize=14)
ax.legend(fontsize=11)
ax.set_xticks([64, 128, 256])
plt.tight_layout()
plt.show()

---
## 7. Final Training & Comparison

Train each model with its best hyperparameters on the **full training set** for 10-15 epochs.

In [ ]:
FINAL_EPOCHS = 15
final_models = {}
final_histories = {}
final_times = {}

for model_type in ['RNN', 'LSTM', 'GRU']:
    print(f'\n{"="*70}')
    print(f'Training final {model_type} on full training data ({len(train_dataset):,} samples)')
    print(f'{"="*70}')
    cfg = best_configs[model_type]
    
    model = TextClassifier(
        model_type=model_type,
        vocab_size=VOCAB_SIZE,
        embed_dim=int(cfg['embed_dim']),
        hidden_size=int(cfg['hidden_size']),
        num_classes=NUM_CLASSES,
        num_layers=int(cfg['num_layers']),
        dropout=cfg['dropout'],
        bidirectional=cfg['bidirectional']
    )
    
    print(f'  Config: embed={int(cfg["embed_dim"])}, hidden={int(cfg["hidden_size"])}, '
          f'layers={int(cfg["num_layers"])}, dropout={cfg["dropout"]}, '
          f'bidir={cfg["bidirectional"]}, lr={cfg["lr"]}')
    print(f'  Parameters: {model.count_parameters():,}')
    
    history, elapsed = train_model(
        model, train_loader, val_loader,
        epochs=FINAL_EPOCHS, lr=cfg['lr'], patience=5, verbose=True
    )
    
    final_models[model_type] = model
    final_histories[model_type] = history
    final_times[model_type] = elapsed
    
    print(f'  Done: {len(history["val_f1"])} epochs, {elapsed:.1f}s, '
          f'best val F1={max(history["val_f1"]):.4f}')

In [ ]:
# Training curves comparison
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Loss
for mt in ['RNN', 'LSTM', 'GRU']:
    h = final_histories[mt]
    axes[0].plot(h['train_loss'], '--', color=COLORS[mt], alpha=0.5, linewidth=1)
    axes[0].plot(h['val_loss'], '-', color=COLORS[mt], label=mt, linewidth=2)
axes[0].set_title('Loss', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CrossEntropy Loss')
axes[0].legend(fontsize=10)

# Accuracy
for mt in ['RNN', 'LSTM', 'GRU']:
    h = final_histories[mt]
    axes[1].plot(h['train_acc'], '--', color=COLORS[mt], alpha=0.5, linewidth=1)
    axes[1].plot(h['val_acc'], '-', color=COLORS[mt], label=mt, linewidth=2)
axes[1].set_title('Accuracy', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=10)

# Macro F1
for mt in ['RNN', 'LSTM', 'GRU']:
    h = final_histories[mt]
    axes[2].plot(h['train_f1'], '--', color=COLORS[mt], alpha=0.5, linewidth=1)
    axes[2].plot(h['val_f1'], '-', color=COLORS[mt], label=mt, linewidth=2)
axes[2].set_title('Macro F1-Score', fontsize=13)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Macro F1')
axes[2].legend(fontsize=10)

plt.suptitle('Training Curves: RNN vs LSTM vs GRU (solid=val, dashed=train)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate all models on test set
all_test_results = {}

for model_type in ['RNN', 'LSTM', 'GRU']:
    preds, labels, probs, metrics = evaluate_model(final_models[model_type], test_loader)
    metrics['Params'] = final_models[model_type].count_parameters()
    metrics['Time (s)'] = f'{final_times[model_type]:.1f}'
    all_test_results[model_type] = {
        'preds': preds, 'labels': labels, 'probs': probs, 'metrics': metrics
    }
    print(f'{model_type} Test Results:')
    print(f'  Accuracy:    {metrics["Accuracy"]:.4f}')
    print(f'  Macro-F1:    {metrics["Macro-F1"]:.4f}')
    print(f'  Weighted-F1: {metrics["Weighted-F1"]:.4f}')
    print()

In [ ]:
# Confusion matrices for all three models
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

for ax, model_type in zip(axes, ['RNN', 'LSTM', 'GRU']):
    preds = all_test_results[model_type]['preds']
    labels = all_test_results[model_type]['labels']
    cm = confusion_matrix(labels, preds)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASS_LIST, yticklabels=CLASS_LIST,
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{model_type} Confusion Matrix', fontsize=13)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    
    # Add accuracy in corner
    acc = all_test_results[model_type]['metrics']['Accuracy']
    ax.text(0.02, 0.98, f'Acc: {acc:.3f}', transform=ax.transAxes,
            verticalalignment='top', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Confusion Matrices on Test Set (World / Sports / Business / Sci-Tech)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 scores — grouped bar chart
per_class_f1 = {}
for model_type in ['RNN', 'LSTM', 'GRU']:
    preds = all_test_results[model_type]['preds']
    labels = all_test_results[model_type]['labels']
    f1_per_class = f1_score(labels, preds, average=None)
    per_class_f1[model_type] = f1_per_class

x = np.arange(NUM_CLASSES)
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, model_type in enumerate(['RNN', 'LSTM', 'GRU']):
    bars = ax.bar(x + i * width, per_class_f1[model_type], width,
                  label=model_type, color=COLORS[model_type], edgecolor='white')
    for bar, val in zip(bars, per_class_f1[model_type]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(CLASS_LIST, fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Per-Class F1 Score Comparison', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.9, color='gray', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# TF-IDF + MultinomialNB baseline
print('Training TF-IDF + MultinomialNB baseline...')
start_nb = time.time()

nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1, 2),
                              sublinear_tf=True, min_df=2)),
    ('clf', MultinomialNB(alpha=0.1))
])

# Use the full original training text (before encoding)
train_texts = (train_df['title'] + ' ' + train_df['description']).values
test_texts = (test_df['title'] + ' ' + test_df['description']).values

nb_pipeline.fit(train_texts, train_labels)
nb_time = time.time() - start_nb

nb_preds = nb_pipeline.predict(test_texts)
nb_acc = accuracy_score(test_labels, nb_preds)
nb_macro_f1 = f1_score(test_labels, nb_preds, average='macro')
nb_weighted_f1 = f1_score(test_labels, nb_preds, average='weighted')

print(f'  Training time: {nb_time:.1f}s')
print(f'  Test Accuracy:    {nb_acc:.4f}')
print(f'  Test Macro-F1:    {nb_macro_f1:.4f}')
print(f'  Test Weighted-F1: {nb_weighted_f1:.4f}')
print(f'\n  TF-IDF vocabulary size: {len(nb_pipeline.named_steps["tfidf"].vocabulary_):,}')

In [ ]:
# Overall comparison table
comparison_data = []

for model_type in ['RNN', 'LSTM', 'GRU']:
    m = all_test_results[model_type]['metrics']
    comparison_data.append({
        'Model': model_type,
        'Accuracy': f'{m["Accuracy"]:.4f}',
        'Macro-F1': f'{m["Macro-F1"]:.4f}',
        'Weighted-F1': f'{m["Weighted-F1"]:.4f}',
        'Params': f'{m["Params"]:,}',
        'Time (s)': m['Time (s)']
    })

# Add NB baseline
comparison_data.append({
    'Model': 'TF-IDF + NB',
    'Accuracy': f'{nb_acc:.4f}',
    'Macro-F1': f'{nb_macro_f1:.4f}',
    'Weighted-F1': f'{nb_weighted_f1:.4f}',
    'Params': 'N/A',
    'Time (s)': f'{nb_time:.1f}'
})

comparison_df = pd.DataFrame(comparison_data).set_index('Model')

print('\n' + '=' * 80)
print('FINAL TEST SET COMPARISON')
print('=' * 80)
comparison_df

In [ ]:
# Summary bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_all = ['RNN', 'LSTM', 'GRU', 'TF-IDF+NB']
colors_all = [COLORS['RNN'], COLORS['LSTM'], COLORS['GRU'], '#95a5a6']

# Accuracy
acc_vals = [all_test_results[m]['metrics']['Accuracy'] for m in ['RNN', 'LSTM', 'GRU']] + [nb_acc]
bars = axes[0].bar(models_all, acc_vals, color=colors_all, edgecolor='white')
for bar, val in zip(bars, acc_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Test Accuracy', fontsize=13)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(min(acc_vals) - 0.05, max(acc_vals) + 0.03)

# Macro F1
f1_vals = [all_test_results[m]['metrics']['Macro-F1'] for m in ['RNN', 'LSTM', 'GRU']] + [nb_macro_f1]
bars = axes[1].bar(models_all, f1_vals, color=colors_all, edgecolor='white')
for bar, val in zip(bars, f1_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[1].set_title('Test Macro-F1', fontsize=13)
axes[1].set_ylabel('Macro F1')
axes[1].set_ylim(min(f1_vals) - 0.05, max(f1_vals) + 0.03)

# Training time
time_vals = [final_times[m] for m in ['RNN', 'LSTM', 'GRU']] + [nb_time]
bars = axes[2].bar(models_all, time_vals, color=colors_all, edgecolor='white')
for bar, val in zip(bars, time_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.0f}s', ha='center', fontweight='bold', fontsize=10)
axes[2].set_title('Training Time', fontsize=13)
axes[2].set_ylabel('Seconds')

plt.suptitle('Final Model Comparison on AG News Test Set', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification reports
for model_type in ['RNN', 'LSTM', 'GRU']:
    preds = all_test_results[model_type]['preds']
    labels = all_test_results[model_type]['labels']
    print(f'\n{"="*60}')
    print(f'{model_type} Classification Report')
    print(f'{"="*60}')
    print(classification_report(labels, preds, target_names=CLASS_LIST, digits=4))

---
## 8. Analysis & Insights

In [ ]:
# Per-class performance deep dive
print('Per-Class Performance Analysis')
print('=' * 80)

for cls_idx, cls_name in enumerate(CLASS_LIST):
    print(f'\n--- {cls_name} (class {cls_idx}) ---')
    for model_type in ['RNN', 'LSTM', 'GRU']:
        preds = all_test_results[model_type]['preds']
        labels = all_test_results[model_type]['labels']
        
        # Metrics for this class
        mask = labels == cls_idx
        cls_acc = (preds[mask] == cls_idx).mean()
        cls_f1 = per_class_f1[model_type][cls_idx]
        
        # Most common misclassification
        misclassified = preds[mask & (preds != cls_idx)]
        if len(misclassified) > 0:
            most_confused = Counter(misclassified).most_common(1)[0]
            confused_class = CLASS_LIST[most_confused[0]]
            confused_count = most_confused[1]
        else:
            confused_class = 'None'
            confused_count = 0
        
        print(f'  {model_type:5s}: Recall={cls_acc:.3f}, F1={cls_f1:.3f}, '
              f'Most confused with: {confused_class} ({confused_count} errors)')

In [ ]:
# Misclassification analysis: show examples of misclassified articles
# Use the best-performing model (LSTM or GRU based on results)
best_model_type = max(['RNN', 'LSTM', 'GRU'],
                      key=lambda m: all_test_results[m]['metrics']['Macro-F1'])
print(f'Misclassification Analysis using {best_model_type} (best model)')
print('=' * 80)

best_preds = all_test_results[best_model_type]['preds']
best_labels = all_test_results[best_model_type]['labels']

# Find misclassified examples
misclassified_mask = best_preds != best_labels
misclassified_idx = np.where(misclassified_mask)[0]

print(f'Total misclassified: {len(misclassified_idx)} / {len(best_labels)} '
      f'({len(misclassified_idx)/len(best_labels)*100:.1f}%)')

# Show some misclassified examples
print(f'\nSample Misclassified Articles:')
print('-' * 80)
np.random.seed(SEED)
sample_misclassified = np.random.choice(misclassified_idx, size=min(8, len(misclassified_idx)), replace=False)

for idx in sample_misclassified:
    true_label = CLASS_LIST[best_labels[idx]]
    pred_label = CLASS_LIST[best_preds[idx]]
    title = test_df.iloc[idx]['title'][:80]
    desc = test_df.iloc[idx]['description'][:100]
    print(f'  True: {true_label:10s} | Predicted: {pred_label:10s}')
    print(f'  Title: {title}')
    print(f'  Desc:  {desc}...')
    print()

In [ ]:
# Title-only vs Full-text comparison
print('Title-Only vs Full-Text Comparison')
print('=' * 60)

# Encode title-only
test_df['title_tokens'] = test_df['title'].apply(preprocess_text)
title_encoded = [encode_tokens(tokens, word2idx, MAX_SEQ_LEN) for tokens in test_df['title_tokens']]
X_test_title = torch.tensor(title_encoded, dtype=torch.long)
title_dataset = TextDataset(X_test_title, y_test)
title_loader = DataLoader(title_dataset, batch_size=BATCH_SIZE)

title_vs_full = []

for model_type in ['RNN', 'LSTM', 'GRU']:
    # Full text (already evaluated)
    full_acc = all_test_results[model_type]['metrics']['Accuracy']
    full_f1 = all_test_results[model_type]['metrics']['Macro-F1']
    
    # Title only
    _, _, _, title_metrics = evaluate_model(final_models[model_type], title_loader)
    title_acc = title_metrics['Accuracy']
    title_f1 = title_metrics['Macro-F1']
    
    title_vs_full.append({
        'Model': model_type,
        'Title-Only Acc': title_acc,
        'Full-Text Acc': full_acc,
        'Acc Diff': full_acc - title_acc,
        'Title-Only F1': title_f1,
        'Full-Text F1': full_f1,
        'F1 Diff': full_f1 - title_f1
    })
    print(f'  {model_type}: Title Acc={title_acc:.4f} vs Full Acc={full_acc:.4f} '
          f'(+{(full_acc-title_acc)*100:.1f}%)')

tvf_df = pd.DataFrame(title_vs_full)

# Plot
x = np.arange(3)
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(x - width/2, tvf_df['Title-Only Acc'], width, label='Title Only', color='#95a5a6')
ax1.bar(x + width/2, tvf_df['Full-Text Acc'], width, label='Title + Description',
        color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']])
ax1.set_xticks(x)
ax1.set_xticklabels(['RNN', 'LSTM', 'GRU'])
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy: Title-Only vs Full-Text', fontsize=13)
ax1.legend()

ax2.bar(x - width/2, tvf_df['Title-Only F1'], width, label='Title Only', color='#95a5a6')
ax2.bar(x + width/2, tvf_df['Full-Text F1'], width, label='Title + Description',
        color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']])
ax2.set_xticks(x)
ax2.set_xticklabels(['RNN', 'LSTM', 'GRU'])
ax2.set_ylabel('Macro F1')
ax2.set_title('Macro-F1: Title-Only vs Full-Text', fontsize=13)
ax2.legend()

plt.suptitle('Impact of Description on Classification Performance', fontsize=14)
plt.tight_layout()
plt.show()

print('\nObservation: Including the description provides additional context and improves accuracy.')
print('Titles alone contain strong topic signals, but descriptions add discriminative details.')

In [ ]:
# Layer depth ablation: 1 layer vs 2 layers vs 3 layers
print('Layer Depth Ablation Study')
print('=' * 60)

# Use subsampled data for speed
layer_depths = [1, 2, 3]
ablation_results = []

for num_layers in layer_depths:
    for model_type in ['RNN', 'LSTM', 'GRU']:
        model = TextClassifier(
            model_type=model_type,
            vocab_size=VOCAB_SIZE,
            embed_dim=128,
            hidden_size=128,
            num_classes=NUM_CLASSES,
            num_layers=num_layers,
            dropout=0.2 if num_layers > 1 else 0.0
        )
        
        history, t = train_model(
            model, sub_loader, val_loader,
            epochs=5, lr=1e-3, verbose=False, patience=3
        )
        
        best_f1 = max(history['val_f1'])
        ablation_results.append({
            'model_type': model_type,
            'num_layers': num_layers,
            'val_f1': best_f1,
            'params': model.count_parameters(),
            'time': t
        })
        print(f'  {model_type} L={num_layers}: val_f1={best_f1:.4f}, '
              f'params={model.count_parameters():,}, time={t:.1f}s')
        del model
    print()

ablation_layer_df = pd.DataFrame(ablation_results)

In [ ]:
# Plot layer depth ablation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for model_type in ['RNN', 'LSTM', 'GRU']:
    subset = ablation_layer_df[ablation_layer_df['model_type'] == model_type]
    ax1.plot(subset['num_layers'], subset['val_f1'], 'o-',
             color=COLORS[model_type], label=model_type, linewidth=2, markersize=8)
    ax2.plot(subset['num_layers'], subset['params'], 'o-',
             color=COLORS[model_type], label=model_type, linewidth=2, markersize=8)

ax1.set_xlabel('Number of Layers', fontsize=12)
ax1.set_ylabel('Val Macro-F1', fontsize=12)
ax1.set_title('Performance vs Layer Depth', fontsize=13)
ax1.legend(fontsize=11)
ax1.set_xticks(layer_depths)

ax2.set_xlabel('Number of Layers', fontsize=12)
ax2.set_ylabel('Parameters', fontsize=12)
ax2.set_title('Parameters vs Layer Depth', fontsize=13)
ax2.legend(fontsize=11)
ax2.set_xticks(layer_depths)

plt.suptitle('Layer Depth Ablation (embed=128, hidden=128)', fontsize=14)
plt.tight_layout()
plt.show()

print('Observations:')
print('- Deeper networks do not always improve performance for text classification.')
print('- 1-2 layers are typically sufficient for this task.')
print('- 3 layers add parameters and training time with diminishing (or negative) returns.')
print('- RNN may suffer more from depth due to vanishing gradients.')

---
## 9. Conclusion

### Summary of Results

| Aspect | RNN | LSTM | GRU | TF-IDF + NB |
|--------|-----|------|-----|-------------|
| **Architecture** | Simple recurrence | 3 gates (forget, input, output) | 2 gates (reset, update) | Bag-of-words |
| **Sequence Modeling** | Weak on long texts | Strong long-range | Good long-range | None (order-agnostic) |
| **Parameters** | Fewest RNN params | Most RNN params | Middle | N/A |
| **Training Speed** | Fastest | Slowest | Middle | Very fast |
| **Expected Performance** | Lower | Highest | Close to LSTM | Competitive baseline |

### Key Findings

1. **LSTM and GRU outperform vanilla RNN** on multi-class text classification, thanks to gating mechanisms that help capture long-range dependencies in news text.

2. **GRU is competitive with LSTM** while using fewer parameters and training faster. For AG News, GRU often matches LSTM accuracy.

3. **TF-IDF + Naive Bayes is a strong baseline**: despite being a simple bag-of-words model, it performs surprisingly well on topic classification because topic keywords are highly discriminative.

4. **Sports is typically the easiest class** to classify due to distinctive vocabulary (game, team, player, score). World and Business can be confused due to overlapping economic/political terms.

5. **Description matters**: Including the description alongside the title provides significant improvement, as it offers more context and discriminative features.

6. **Deeper is not always better**: 1-2 layer networks are sufficient for this task. Adding more layers increases parameters and training time without consistent gains.

### Extensions
- **Pre-trained embeddings**: Use GloVe or Word2Vec to initialize the embedding layer for better word representations
- **Attention mechanisms**: Add self-attention to focus on the most relevant parts of the text
- **Transformer baselines**: Compare with BERT or DistilBERT fine-tuning
- **Character-level models**: Handle out-of-vocabulary words and typos
- **Multi-task learning**: Jointly predict title category and description category
- **Data augmentation**: Synonym replacement, back-translation to increase training diversity

In [ ]:
# Final summary visualization
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

models_plot = ['RNN', 'LSTM', 'GRU']
colors_plot = [COLORS[m] for m in models_plot]

# 1. Accuracy
acc_vals = [all_test_results[m]['metrics']['Accuracy'] for m in models_plot]
bars = axes[0].bar(models_plot, acc_vals, color=colors_plot, edgecolor='white')
for bar, val in zip(bars, acc_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', fontweight='bold')
axes[0].set_title('Test Accuracy', fontsize=13)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(min(acc_vals) - 0.05, 1.0)

# 2. Macro-F1
f1_vals = [all_test_results[m]['metrics']['Macro-F1'] for m in models_plot]
bars = axes[1].bar(models_plot, f1_vals, color=colors_plot, edgecolor='white')
for bar, val in zip(bars, f1_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', fontweight='bold')
axes[1].set_title('Test Macro-F1', fontsize=13)
axes[1].set_ylabel('Macro F1')
axes[1].set_ylim(min(f1_vals) - 0.05, 1.0)

# 3. Training Time
time_vals = [final_times[m] for m in models_plot]
bars = axes[2].bar(models_plot, time_vals, color=colors_plot, edgecolor='white')
for bar, val in zip(bars, time_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.0f}s', ha='center', fontweight='bold')
axes[2].set_title('Training Time', fontsize=13)
axes[2].set_ylabel('Seconds')

# 4. Parameters
param_vals = [final_models[m].count_parameters() for m in models_plot]
bars = axes[3].bar(models_plot, param_vals, color=colors_plot, edgecolor='white')
for bar, val in zip(bars, param_vals):
    axes[3].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=9)
axes[3].set_title('Parameters', fontsize=13)
axes[3].set_ylabel('Trainable Parameters')

plt.suptitle('Final Summary: RNN vs LSTM vs GRU on AG News (4-Class Classification)', fontsize=14)
plt.tight_layout()
plt.show()

print('Notebook complete!')
print('Next: Case Study 6 for sequence-to-sequence or more advanced NLP tasks.')